In [88]:
import pandas as pd
import numpy as np
from dateutil import parser
import psycopg

In [89]:
df=pd.read_csv("~/health_pipline/healthcare_dataset.csv")

In [90]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 55500 entries, 0 to 55499
Data columns (total 15 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Name                55500 non-null  str    
 1   Age                 55500 non-null  int64  
 2   Gender              55500 non-null  str    
 3   Blood Type          55500 non-null  str    
 4   Medical Condition   55500 non-null  str    
 5   Date of Admission   55500 non-null  str    
 6   Doctor              55500 non-null  str    
 7   Hospital            55500 non-null  str    
 8   Insurance Provider  55500 non-null  str    
 9   Billing Amount      55500 non-null  float64
 10  Room Number         55500 non-null  int64  
 11  Admission Type      55500 non-null  str    
 12  Discharge Date      55500 non-null  str    
 13  Medication          55500 non-null  str    
 14  Test Results        55500 non-null  str    
dtypes: float64(1), int64(2), str(12)
memory usage: 6.4 MB


In [91]:
df.dtypes


Name                      str
Age                     int64
Gender                    str
Blood Type                str
Medical Condition         str
Date of Admission         str
Doctor                    str
Hospital                  str
Insurance Provider        str
Billing Amount        float64
Room Number             int64
Admission Type            str
Discharge Date            str
Medication                str
Test Results              str
dtype: object

In [92]:
df.isnull().sum()


Name                  0
Age                   0
Gender                0
Blood Type            0
Medical Condition     0
Date of Admission     0
Doctor                0
Hospital              0
Insurance Provider    0
Billing Amount        0
Room Number           0
Admission Type        0
Discharge Date        0
Medication            0
Test Results          0
dtype: int64

In [93]:
df.duplicated().sum()

np.int64(534)

In [94]:
df.describe()


,Age,Billing Amount,Room Number
count,55500.000000,55500.000000,55500.000000
mean,51.539459,25539.316097,301.134829
std,19.602454,14211.454431,115.243069
min,13.000000,-2008.492140,101.000000
25%,35.000000,13241.224652,202.000000
50%,52.000000,25538.069376,302.000000
75%,68.000000,37820.508436,401.000000
max,89.000000,52764.276736,500.000000


In [95]:
df["Name"].sample(20)
df["Gender"].unique()
df["Blood Type"].unique()
df["Medical Condition"].unique()
df["Doctor"].sample(20)
df["Hospital"].sample(20)
df["Insurance Provider"].unique()
df["Admission Type"].unique()
df["Medication"].unique()
df["Test Results"].unique()

<StringArray>
['Normal', 'Inconclusive', 'Abnormal']
Length: 3, dtype: str

In [96]:
#remove the duplicates 
df=df.drop_duplicates()
df.duplicated().sum()


np.int64(0)

In [97]:
#Change the categorical columns to lowercase
df_col=["Name","Gender","Blood Type","Medical Condition","Doctor","Hospital","Insurance Provider",\
    "Admission Type","Medication","Test Results"]

for col in df_col:
    df[col]=df[col].str.lower().str.strip()

In [98]:
#Handling the negaitve billing amounts
df["Billing Amount"]=df["Billing Amount"].abs()

In [99]:
#Changing the Date columns to date types
df["Date of Admission"]=pd.to_datetime(df["Date of Admission"])
df["Discharge Date"]=pd.to_datetime(df["Discharge Date"])

In [100]:
df.dtypes

Name                             str
Age                            int64
Gender                           str
Blood Type                       str
Medical Condition                str
Date of Admission     datetime64[us]
Doctor                           str
Hospital                         str
Insurance Provider               str
Billing Amount               float64
Room Number                    int64
Admission Type                   str
Discharge Date        datetime64[us]
Medication                       str
Test Results                     str
dtype: object

In [101]:
#Handling patient name (firstName,lastName)
df[["patient_title","name"]]=df["Name"].str.extract(r'^(?:([A-Za-z]+)\.\s*)?(.*)')
df[["first","middle","last"]]=df["Name"].str.extract(r'([A-Za-z]+)\s+([A-Za-z]\.)?\s*([A-Za-z]+)')
df["Name"]=df["first"] + " " + df["last"]
# drop the reduntant columns 
df=df.drop(["name","first","middle","last"],axis=1)


In [102]:
#Handling Doctor name (firstName,lastName)
df[["doctor_title","name"]]=df["Name"].str.extract(r'^(?:([A-Za-z]+)\.\s*)?(.*)')
df[["first","middle","last"]]=df["Name"].str.extract(r'([A-Za-z]+)\s+([A-Za-z]\.)?\s*([A-Za-z]+)')
df["Doctor"]=df["first"] + " " + df["last"]
# drop the reduntant columns 
df=df.drop(["name","first","middle","last","doctor_title"],axis=1)

In [103]:
#handling Hospital names
df["Hospital"] = df["Hospital"].str.replace(r"\band\b|,", "", regex=True)
df["Hospital"] = df["Hospital"].str.replace(r"\s+|-", " ", regex=True)
df["Hospital"] = df["Hospital"].str.strip()

In [104]:
#handling column names
df.columns=df.columns.str.lower().str.replace(' ','_')
df

,name,age,gender,blood_type,medical_condition,date_of_admission,doctor,hospital,insurance_provider,billing_amount,room_number,admission_type,discharge_date,medication,test_results,patient_title
0,bobby jackson,30,male,b-,cancer,2024-01-31,bobby jackson,sons miller,blue cross,18856.281306,328,urgent,2024-02-02,paracetamol,normal,NaN
1,leslie terry,62,male,a+,obesity,2019-08-20,leslie terry,kim inc,medicare,33643.327287,265,emergency,2019-08-26,ibuprofen,inconclusive,NaN
2,danny smith,76,female,a-,obesity,2022-09-22,danny smith,cook plc,aetna,27955.096079,205,emergency,2022-10-07,aspirin,normal,NaN
3,andrew watts,28,female,o+,diabetes,2020-11-18,andrew watts,hernandez rogers vang,medicare,37909.782410,450,elective,2020-12-18,ibuprofen,abnormal,NaN
4,adrienne bell,43,female,ab+,cancer,2022-09-19,adrienne bell,white white,aetna,14238.317814,458,urgent,2022-10-09,penicillin,abnormal,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
55495,elizabeth jackson,42,female,o+,asthma,2020-08-16,elizabeth jackson,jones thompson,blue cross,2650.714952,417,elective,2020-09-15,penicillin,abnormal,NaN
55496,kyle perez,61,female,ab-,obesity,2020-01-23,kyle perez,tucker moyer,cigna,31457.797307,316,elective,2020-02-01,aspirin,normal,NaN
55497,heather wang,38,female,b+,hypertension,2020-07-13,heather wang,mahoney johnson vasquez,unitedhealthcare,27620.764717,347,urgent,2020-08-10,ibuprofen,abnormal,NaN
55498,jennifer jones,43,male,o-,arthritis,2019-05-25,jennifer jones,jackson todd castro,medicare,32451.092358,321,elective,2019-05-31,ibuprofen,abnormal,NaN


In [105]:
df.to_csv("~/health_pipline/healthcare_cleaned.csv",index=False)

In [106]:
conn_str = "dbname=medical user=postgres password=postgres host=localhost port=5432"
doctor_df = df[["name"]].drop_duplicates()
hospital_df = df[["hospital"]].drop_duplicates()
patient_df = df[["name","age", "gender", "blood_type", "insurance_provider","patient_title"]].drop_duplicates(subset=["name","age","gender","blood_type","insurance_provider"])

with psycopg.connect(conn_str) as conn:
    with conn.cursor() as cur:

        # doctor
        with cur.copy("COPY doctor (name) FROM STDIN") as copy:
            for row in doctor_df.itertuples(index=False, name=None):
                copy.write_row(row)

        # hospital
        with cur.copy("COPY hospital (name) FROM STDIN") as copy:
            for row in hospital_df.itertuples(index=False, name=None):
                copy.write_row(row)

        # patient
        with cur.copy("COPY patient (name,age, gender, blood_type, insurance_provider,patient_title) FROM STDIN") as copy:
            for row in patient_df.itertuples(index=False, name=None):
                copy.write_row(row)
        

In [107]:
with psycopg.connect(conn_str) as conn:
    patient=pd.read_sql_query("""select * from patient""",conn)
    doctor=pd.read_sql_query("""select * from doctor""",conn)
    hospital=pd.read_sql_query("""select * from hospital""",conn)
   

/tmp/ipykernel_4051/1860832911.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  patient=pd.read_sql_query("""select * from patient""",conn)
/tmp/ipykernel_4051/1860832911.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  doctor=pd.read_sql_query("""select * from doctor""",conn)
/tmp/ipykernel_4051/1860832911.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  hospital=pd.read_sql_query("""select * from hospital""",conn)


In [108]:
doctor = doctor.rename(
    columns={
        "doctor_id": "doctor_id",
        "name": "doctor_name"
    }
)

hospital = hospital.rename(
    columns={
        "hospital_id": "hospital_id",
        "name": "hospital_name"
    }
)

patient = patient.rename(
    columns={
        "patient_id": "patient_id",
        "name": "patient_name"
    }
)


In [109]:
df = df.merge(
    doctor,
    left_on="doctor",
    right_on="doctor_name",
    how="left"
)

In [110]:
df = df.merge(
    hospital,
    left_on="hospital",
    right_on="hospital_name",
    how="left"
)

In [111]:
df

,name,age,gender,blood_type,medical_condition,date_of_admission,doctor,hospital,insurance_provider,billing_amount,room_number,admission_type,discharge_date,medication,test_results,patient_title,doctor_id,doctor_name,hospital_id,hospital_name
0,bobby jackson,30,male,b-,cancer,2024-01-31,bobby jackson,sons miller,blue cross,18856.281306,328,urgent,2024-02-02,paracetamol,normal,NaN,1,bobby jackson,1,sons miller
1,leslie terry,62,male,a+,obesity,2019-08-20,leslie terry,kim inc,medicare,33643.327287,265,emergency,2019-08-26,ibuprofen,inconclusive,NaN,2,leslie terry,2,kim inc
2,danny smith,76,female,a-,obesity,2022-09-22,danny smith,cook plc,aetna,27955.096079,205,emergency,2022-10-07,aspirin,normal,NaN,3,danny smith,3,cook plc
3,andrew watts,28,female,o+,diabetes,2020-11-18,andrew watts,hernandez rogers vang,medicare,37909.782410,450,elective,2020-12-18,ibuprofen,abnormal,NaN,4,andrew watts,4,hernandez rogers vang
4,adrienne bell,43,female,ab+,cancer,2022-09-19,adrienne bell,white white,aetna,14238.317814,458,urgent,2022-10-09,penicillin,abnormal,NaN,5,adrienne bell,5,white white
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
54961,elizabeth jackson,42,female,o+,asthma,2020-08-16,elizabeth jackson,jones thompson,blue cross,2650.714952,417,elective,2020-09-15,penicillin,abnormal,NaN,21000,elizabeth jackson,21662,jones thompson
54962,kyle perez,61,female,ab-,obesity,2020-01-23,kyle perez,tucker moyer,cigna,31457.797307,316,elective,2020-02-01,aspirin,normal,NaN,803,kyle perez,793,tucker moyer
54963,heather wang,38,female,b+,hypertension,2020-07-13,heather wang,mahoney johnson vasquez,unitedhealthcare,27620.764717,347,urgent,2020-08-10,ibuprofen,abnormal,NaN,1866,heather wang,1839,mahoney johnson vasquez
54964,jennifer jones,43,male,o-,arthritis,2019-05-25,jennifer jones,jackson todd castro,medicare,32451.092358,321,elective,2019-05-31,ibuprofen,abnormal,NaN,2330,jennifer jones,21891,jackson todd castro


In [112]:
df = df.merge(
    patient,
    left_on=[
        "name",
        "age",
        "blood_type",
        "insurance_provider"
    ],
    right_on=[
        "patient_name",
        "age",
        "blood_type",
        "insurance_provider"
    ],
    how="left"
)

In [113]:
df

,name,age,gender_x,blood_type,medical_condition,date_of_admission,doctor,hospital,insurance_provider,billing_amount,...,test_results,patient_title_x,doctor_id,doctor_name,hospital_id,hospital_name,patient_id,patient_name,gender_y,patient_title_y
0,bobby jackson,30,male,b-,cancer,2024-01-31,bobby jackson,sons miller,blue cross,18856.281306,...,normal,NaN,1,bobby jackson,1,sons miller,1,bobby jackson,male,nan
1,leslie terry,62,male,a+,obesity,2019-08-20,leslie terry,kim inc,medicare,33643.327287,...,inconclusive,NaN,2,leslie terry,2,kim inc,2,leslie terry,male,nan
2,danny smith,76,female,a-,obesity,2022-09-22,danny smith,cook plc,aetna,27955.096079,...,normal,NaN,3,danny smith,3,cook plc,3,danny smith,female,nan
3,andrew watts,28,female,o+,diabetes,2020-11-18,andrew watts,hernandez rogers vang,medicare,37909.782410,...,abnormal,NaN,4,andrew watts,4,hernandez rogers vang,4,andrew watts,female,nan
4,adrienne bell,43,female,ab+,cancer,2022-09-19,adrienne bell,white white,aetna,14238.317814,...,abnormal,NaN,5,adrienne bell,5,white white,5,adrienne bell,female,nan
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
54971,elizabeth jackson,42,female,o+,asthma,2020-08-16,elizabeth jackson,jones thompson,blue cross,2650.714952,...,abnormal,NaN,21000,elizabeth jackson,21662,jones thompson,54959,elizabeth jackson,female,nan
54972,kyle perez,61,female,ab-,obesity,2020-01-23,kyle perez,tucker moyer,cigna,31457.797307,...,normal,NaN,803,kyle perez,793,tucker moyer,54960,kyle perez,female,nan
54973,heather wang,38,female,b+,hypertension,2020-07-13,heather wang,mahoney johnson vasquez,unitedhealthcare,27620.764717,...,abnormal,NaN,1866,heather wang,1839,mahoney johnson vasquez,54961,heather wang,female,nan
54974,jennifer jones,43,male,o-,arthritis,2019-05-25,jennifer jones,jackson todd castro,medicare,32451.092358,...,abnormal,NaN,2330,jennifer jones,21891,jackson todd castro,54962,jennifer jones,male,nan


In [114]:
df_visit=df[["patient_id","doctor_id","hospital_id","date_of_admission","discharge_date","admission_type","medical_condition","billing_amount","medication","test_results","room_number"]]

In [118]:
with psycopg.connect(conn_str) as conn:
    with conn.cursor() as cur:
        with cur.copy("COPY visit (patient_id,doctor_id,hospital_id,date_of_admission,discharge_date,admission_type,medical_condition,billing_amount,medication,test_results,room_number) FROM STDIN") as copy:
            for row in df_visit.itertuples(index=False, name=None):
                copy.write_row(row)

In [120]:
df.sample(50)

,name,age,gender_x,blood_type,medical_condition,date_of_admission,doctor,hospital,insurance_provider,billing_amount,...,test_results,patient_title_x,doctor_id,doctor_name,hospital_id,hospital_name,patient_id,patient_name,gender_y,patient_title_y
36634,paul pugh,55,female,b-,asthma,2020-03-16,paul pugh,mcdonald maldonado guerra,aetna,27667.172828,...,abnormal,NaN,3135,paul pugh,29701,mcdonald maldonado guerra,36627,paul pugh,female,nan
46409,vanessa thomas,74,female,b-,asthma,2020-01-26,vanessa thomas,ltd beasley,aetna,35355.361005,...,inconclusive,NaN,37187,vanessa thomas,36693,ltd beasley,46400,vanessa thomas,female,nan
22813,jill morales,52,female,ab-,obesity,2022-07-02,jill morales,inc owen,cigna,2248.820001,...,inconclusive,NaN,19929,jill morales,19423,inc owen,22809,jill morales,female,nan
50335,april jones,38,female,b-,obesity,2019-11-17,april jones,inc rodriguez,medicare,15528.373245,...,abnormal,NaN,12524,april jones,2629,inc rodriguez,50325,april jones,female,nan
46393,timothy anderson,47,male,b+,obesity,2021-05-09,timothy anderson,knight miles white,medicare,47674.801070,...,inconclusive,NaN,17717,timothy anderson,36682,knight miles white,46384,timothy anderson,male,nan
40994,carolyn ryan,39,female,a+,diabetes,2023-02-27,carolyn ryan,bryant inc,medicare,19496.264691,...,inconclusive,NaN,33406,carolyn ryan,11029,bryant inc,40987,carolyn ryan,female,nan
44290,daniel robinson,37,male,a+,asthma,2020-12-15,daniel robinson,flores hamilton,blue cross,23636.717585,...,inconclusive,NaN,19582,daniel robinson,35197,flores hamilton,44282,daniel robinson,male,nan
3087,shannon ayala,61,female,b-,arthritis,2020-07-21,shannon ayala,skinner simmons scott,blue cross,21717.020299,...,abnormal,NaN,3017,shannon ayala,2958,skinner simmons scott,3088,shannon ayala,female,nan
52270,edwin banks,55,female,a+,asthma,2023-03-31,edwin banks,ayers king smith,cigna,34597.392255,...,normal,NaN,21917,edwin banks,21395,ayers king smith,52259,edwin banks,female,nan
42267,lisa ortega,59,female,b-,hypertension,2021-04-26,lisa ortega,vang inc,blue cross,34663.070573,...,abnormal,NaN,34304,lisa ortega,30908,vang inc,42260,lisa ortega,female,nan
